# ITELv5 - SECCIÓN 8: EXPORTACIÓN DEL MODELO FINAL

En esta sección, usamos los mejores hiperparámetros encontrados en la Sección 6 para entrenar un modelo MLP final y un Scaler final sobre *todo* el conjunto de datos (3,730 audios).

Estos archivos (`final_mlp_model.pt` y `final_mlp_scaler.joblib`) son los "artefactos" de producción que se usarán para la inferencia en un servidor o API.

In [1]:
import os
from pathlib import Path

# Asegurar que el CWD sea la carpeta de notebooks
# (IntelliJ puede configurar un CWD distinto al de Jupyter Lab)
_anchor = Path('ITELv5_01_CONFIGURACION.ipynb')
if not _anchor.exists():
    for _candidate in Path.cwd().rglob('ITELv5_01_CONFIGURACION.ipynb'):
        os.chdir(_candidate.parent)
        break
print(f'📂 CWD: {Path(".").resolve()}')


📂 CWD: C:\Users\USER\Documents\GitHub\ITEL\research\notebooks


In [2]:
# 8.5.1 - Instalar ONNX
import sys
!{sys.executable} -m pip install onnx onnxruntime

print("\n✅ Paquetes 'onnx' y 'onnxruntime' instalados.")


✅ Paquetes 'onnx' y 'onnxruntime' instalados.


In [3]:
# 8.1 - Configuración e importaciones
%run "./ITELv5_01_CONFIGURACION.ipynb"

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import joblib # Usaremos joblib para guardar el scaler
from sklearn.preprocessing import StandardScaler

# Asegurarse de que las métricas (usadas en HPO) están importadas si se corre 06
try:
    from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, accuracy_score
except ImportError:
    pass

print("✅ Entorno de exportación listo.")

🔍 Verificando estructura de carpetas...
  ✅ Datos_Brutos: 1680 WAV Healthy  |  2173 WAV Patients
  ✅ Todas las dependencias disponibles.

📁 BASE_DIR : C:\Users\USER\Documents\GitHub\ITEL\research
📂 DATA_DIR : C:\Users\USER\Documents\GitHub\ITEL\research\Datos_Brutos
🖥️  GPU      : False
Rutas del proyecto ITELv5:
  Datos_Brutos    : C:\Users\USER\Documents\GitHub\ITEL\research\Datos_Brutos
  Procesados      : C:\Users\USER\Documents\GitHub\ITEL\research\processed_data
  Modelos         : C:\Users\USER\Documents\GitHub\ITEL\assets\models
  Reportes CSV    : C:\Users\USER\Documents\GitHub\ITEL\research\reports\CSV
  Reportes Figuras: C:\Users\USER\Documents\GitHub\ITEL\research\reports\Figuras
  Logs            : C:\Users\USER\Documents\GitHub\ITEL\research\logs

🔍 Healthy  : 308 subrutas de audio
🔍 Patients : 378 subrutas de audio

✅ Configuración ITELv5 lista.
✅ Entorno de exportación listo.


In [4]:
# 8.2 - Definir la Arquitectura del Modelo MLP
# (Copiado de 06_Training)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(nn.Module):
    def __init__(self, in_dim=4608, hidden=512, dropout=0.3, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, num_classes)
        )
    def forward(self, x):
        return self.net(x)

print(f"✅ Definición MLP lista. Usando device: {device}")

✅ Definición MLP lista. Usando device: cpu


In [5]:
# 8.3 - Cargar 10% de datos y los 5 modelos base

print('Cargando 10% para stacking...')

emb_dir = ITELConfig.PROCESADOS_DIR
X_all   = np.load(emb_dir / 'embeddings_v5.npy')
info_all = pd.read_csv(emb_dir / 'embeddings_info_v5.csv')

idx_final_path = emb_dir / 'idx_final_v5.npy'
if not idx_final_path.exists():
    raise FileNotFoundError('Ejecuta primero el Notebook 06 para generar el split 80/10/10.')

idx_final  = np.load(idx_final_path)
X_10       = X_all[idx_final]          # embeddings del 10%
info_10    = info_all.iloc[idx_final].reset_index(drop=True)
label_map  = {'Healthy': 0, 'Patients': 1}
y_10       = info_10['categoria'].map(label_map).values

# Cargar HParams y Scaler del CV (entrenados sobre el 80%)
hparams_path = ITELConfig.MODELS_DIR / 'mlp_hparams_v5.json'
with open(hparams_path) as f:
    hparams = json.load(f)

scaler_data  = np.load(ITELConfig.MODELS_DIR / 'linear_head_scaler.npz')
from sklearn.preprocessing import StandardScaler
cv_scaler    = StandardScaler()
cv_scaler.mean_  = scaler_data['mean']
cv_scaler.scale_ = scaler_data['scale']

# Escalar el 10% con el scaler del CV (NO re-entrenar el scaler)
X_10_scaled = cv_scaler.transform(X_10)
X_10_t      = torch.tensor(X_10_scaled, dtype=torch.float32).to(device)

# Cargar los 5 modelos del CV
N_SPLITS = 5
base_models = []
for fold in range(1, N_SPLITS + 1):
    m = MLP(in_dim=X_10.shape[1],
            hidden=hparams['hidden'],
            dropout=hparams['dropout']).to(device)
    m.load_state_dict(torch.load(
        ITELConfig.MODELS_DIR / f'itelv5_mlp_fold_{fold}.pt',
        map_location=device))
    m.eval()
    base_models.append(m)

print(f'  ✅ {len(base_models)} modelos base cargados')
print(f'  ✅ 10% cargado: {len(X_10)} audios  |  Healthy: {(y_10==0).sum()}  Patients: {(y_10==1).sum()}')


Cargando 10% para stacking...
  ✅ 5 modelos base cargados
  ✅ 10% cargado: 373 audios  |  Healthy: 180  Patients: 193


In [6]:
# 8.4 - Los 5 modelos predicen sobre el 10%
# Resultado: matriz de stacking (373 × 10)
# Cada fila = 1 audio | Cada par de columnas = probabilidades de un modelo base

print('Generando predicciones de los 5 modelos sobre el 10%...')

probs_list = []
with torch.inference_mode():
    for fold_idx, model in enumerate(base_models):
        logits = model(X_10_t)                          # (N, 2)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()  # (N, 2)
        probs_list.append(probs)
        acc = (logits.argmax(1).cpu().numpy() == y_10).mean()
        print(f'  Modelo fold {fold_idx+1}: accuracy sobre 10% = {acc:.4f}')

# Stack horizontal: (N, 5×2) = (N, 10)
# Columnas: [m1_healthy, m1_patients, m2_healthy, m2_patients, ...]
X_stack = np.hstack(probs_list)   # shape: (373, 10)

print(f'\n  ✅ Matriz de stacking generada: {X_stack.shape}')
print(f'     → {N_SPLITS} modelos × 2 probabilidades = {X_stack.shape[1]} features')
print(f'     → El modelo final aprende a combinar estas opiniones')


Generando predicciones de los 5 modelos sobre el 10%...
  Modelo fold 1: accuracy sobre 10% = 0.9062
  Modelo fold 2: accuracy sobre 10% = 0.8954
  Modelo fold 3: accuracy sobre 10% = 0.9062
  Modelo fold 4: accuracy sobre 10% = 0.9142
  Modelo fold 5: accuracy sobre 10% = 0.9062

  ✅ Matriz de stacking generada: (373, 10)
     → 5 modelos × 2 probabilidades = 10 features
     → El modelo final aprende a combinar estas opiniones


In [7]:
# 8.5 - Entrenar el modelo final (meta-learner)
# Entrena sobre X_stack (las 10 probabilidades de los 5 modelos)
# Es una Regresión Logística: simple, interpretable, y difícil de sobreajustar
# con solo 10 features.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score

print('Entrenando meta-modelo (Regresión Logística sobre stacking)...')

meta_model = LogisticRegression(max_iter=1000, random_state=ITELConfig.SEED, C=1.0)
meta_model.fit(X_stack, y_10)

# Evaluar sobre el mismo 10% (referencia — el verdadero test es el holdout manual)
y_pred_meta = meta_model.predict(X_stack)
print(f'\n  Accuracy meta-modelo (10%): {accuracy_score(y_10, y_pred_meta):.4f}')
print(f'  F1                        : {f1_score(y_10, y_pred_meta):.4f}')
print(f'  Recall                    : {recall_score(y_10, y_pred_meta):.4f}')
print(f'  ⚠️  Estas métricas son sobre datos de entrenamiento.')
print(f'     El verdadero test es el holdout manual (Notebook 07).')

# Guardar el meta-modelo
meta_path = ITELConfig.MODELS_DIR / 'final_meta_model_v5.joblib'
joblib.dump(meta_model, meta_path)
print(f'\n  💾 Meta-modelo guardado: {meta_path}')


Entrenando meta-modelo (Regresión Logística sobre stacking)...

  Accuracy meta-modelo (10%): 0.9088
  F1                        : 0.9119
  Recall                    : 0.9119
  ⚠️  Estas métricas son sobre datos de entrenamiento.
     El verdadero test es el holdout manual (Notebook 07).

  💾 Meta-modelo guardado: C:\Users\USER\Documents\GitHub\ITEL\assets\models\final_meta_model_v5.joblib


In [8]:
# 8.6 - Exportar el Modelo Final a ONNX (Opcional pero recomendado)

print("Exportando modelo final a formato ONNX...")

try:
    # 1. Asegurarse de que el modelo esté en modo evaluación
    final_model.eval()

    # 2. Crear una entrada de ejemplo (dummy input)
    # Necesita tener el shape correcto: (1, 4608)
    dummy_input = X_scaled_t[0].unsqueeze(0).to(device) 
    
    onnx_path = ITELConfig.MODELS_DIR / 'final_mlp_model.onnx'

    # 3. Exportar
    torch.onnx.export(
        final_model,               # El modelo a exportar
        dummy_input,             # Una entrada de ejemplo
        onnx_path,               # Dónde guardar el archivo
        input_names=['input_embedding'], # Nombre de la entrada
        output_names=['output_logits'], # Nombre de la salida
        dynamic_axes={'input_embedding': {0: 'batch_size'}, # Permitir batch_size variable
                      'output_logits': {0: 'batch_size'}},
        opset_version=11 # Versión de ONNX
    )
    
    print(f"✅ Modelo exportado a ONNX exitosamente en: {onnx_path}")
    print("Esta versión .onnx es ideal para servidores de inferencia optimizados (como ONNX Runtime).")

except Exception as e:
    print(f"⚠️ Ocurrió un error al exportar a ONNX: {e}")

Exportando modelo final a formato ONNX...
⚠️ Ocurrió un error al exportar a ONNX: name 'final_model' is not defined


# 8.7 - Resumen de Exportación

¡Exportación completada!

Para usar tu modelo en una página web (mediante una API de Python como Flask o FastAPI), necesitarás los siguientes componentes en tu servidor:

1.  **El Modelo:** `Modelos/final_mlp_model.pt` (para PyTorch) o `Modelos/final_mlp_model.onnx` (para ONNX Runtime).
2.  **El Scaler:** `Modelos/final_mlp_scaler.joblib` (para Scikit-learn).
3.  **Las Funciones:** Las definiciones de `procesar_audio()` (de 04) y `extraer_embedding()` (de 05), junto con sus dependencias (HuBERT, librosa, etc.).

Tu pipeline de inferencia en el servidor será:
**WAV (Usuario)** -> `procesar_audio()` -> `extraer_embedding()` -> `final_scaler.transform()` -> `final_model.forward()` -> **JSON (Resultado)**